### Notebook to proceed with the following steps once the yolo model has been trained. 
Will run manually here sequences of various scripts but mainly the boat_utils.testing.py

Utility functions for training/validation pipeline.  
Includes: 

    - prepare: Prepare the images for segmentation
    - segment: Segment the images
    - run_detection: Run the YoloV5 detection
    - backwards_annotation_AF: Generate labelme style annotations from the classifications
    - compare_detections_to_ground_truth: Match up labels and detections, compare them, and save the results
    - confusion_matrix_AF: Summarize the results of the comparison
    - plot_waterholes: Plot a comparison of my labels vs the model detection of waterholes on single stitched back together images. 

Load first all the configs and required packs:

In [ ]:

import os
import shutil
import yaml 
import argparse
import os.path as path
import scipy.cluster
import scipy.spatial
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import scipy
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from tqdm import tqdm
import torch
import stat
import shutil
from datetime import datetime

from counting_boats.boat_utils.config import cfg
from counting_boats.boat_utils import image_cutting_support as ics
from counting_boats.boat_utils import heatmap as hm

import counting_boats.boat_utils.classifier
# cluster, process_clusters, read_classifications, pixel2latlong

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


If needed, extract the tif files from the zip files obtained from Planet:

In [ ]:
import os
import yaml

import counting_boats.boat_utils.planet_utils

counting_boats.boat_utils.planet_utils.extract_zip_AF(r'D:\Waterholes_project\counting_waterholes\deployment\zips\mimal_full1_20240607_psscene_analytic_sr_udm2.zip',aoi='mimal_full1', date='20240607', cfg="config_train_Drive.yaml")

Then we can select manually which function we want to run. 
First, from the testing aoi tif file, we need to prepare the padded png image using the testing.prepare():

In [ ]:
# Run preparation:
import counting_boats.boat_utils.testing


counting_boats.boat_utils.testing.prepare("D:\Waterholes_project\counting_waterholes\deployment", "config_deploy_Drive.yaml")

Then I need to use the created png to label it with labelme. This will allow us to compare my annotation to the detection of the trained model i.e. test the model. 

Once the manual annotation is done, we can apply the segmentation used from the testing.segment().  
Running time is quite long because of the post-segmentation segregation by date and image for the images + labels (moving them between folders). This should definetly be fixed goign froward in this project to gain processing time. 

The padded pngs and the json files should both be stored in the pngs subfolder of the testing folder. 

In [ ]:
#run segmentation without spliting 80% of the images for validation!
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.segment(r"D:/Waterholes_project/counting_waterholes/deployment", "config_deploy_Drive.yaml")

If you run into an error of folders already created and the image(s) is segmented but wasn't moved to the destination folder, use this function to move your image files around modifying your source and destination folders bellow:

In [ ]:
import os
import shutil

def move_png_files(source_folder, destination_folder):
    # Ensure the destination folder exists
    os.makedirs(destination_folder, exist_ok=True)
    
    # Counter for moved files
    moved_files = 0
    
    # Iterate through all files in the source folder
    for filename in os.listdir(source_folder):
        # Check if the file ends with .png (case-insensitive)
        if filename.lower().endswith('.png'):
            # Create full file paths
            source_path = os.path.join(source_folder, filename)
            destination_path = os.path.join(destination_folder, filename)
            
            # Move the file
            shutil.move(source_path, destination_path)
            moved_files += 1
    
    print(f"Moved {moved_files} PNG files to {destination_folder}")

# Specify the source and destination folders
#To Change!:
source_folder = r'D:\Waterholes_project\counting_waterholes\deployment\segmented_images'
destination_folder = r'D:\Waterholes_project\counting_waterholes\deployment\segmented_images\07_06_2024\20240607_mimalfull1'

# Call the function to move PNG files
move_png_files(source_folder, destination_folder)

Using those segmented labelled images, we can run the detection of waterholes using the trained model, and compare my label with the detection of the model. 

Debugging section to recognise and work well with the GPU:

Note to user: Need to update the torchvision to match the cuda (GPU) version. using the 'nvidia-smi' command, you get the cuda version (my case: 11) so I need to get a version of torch and torchaudio with to 11.xx. Need to 'pip uninstall torch torchvision', and then install the correct version, in my case: 'pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113'  
Other version to be found on this website: https://pytorch.org/get-started/previous-versions/

The testing.segment function and run_detection work to use the segmented images folders grouped per date. left as it is for now but just something to bear in mind!  

In [ ]:
#check of the GPU found or not? Just for debugging
torch.cuda.is_available()

Need to delete the classification folder in that testing directory in case your run it and encouter an error and run it again. The function is not able to overwrite the folder. Keep an eye out to delete it before running it. 

In [ ]:
#run detection on my testing 
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.run_detection(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")
# counting_boats.boat_utils.testing.run_detection(r"D:/Waterholes_project/counting_waterholes/deployment", "config_deploy_Drive.yaml")

Use the detection output of the model on my testing images to produce labelme style annotation using the backwards_annotation():  

In [ ]:
#run the annotation of the images using the detection of the model:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.backwards_annotation_AF(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")
# counting_boats.boat_utils.testing.backwards_annotation_AF(r"D:/Waterholes_project/counting_waterholes/deployment", "config_deploy_Drive.yaml")

The backward annotation produces the auto label json file which you can open manually in labelme to check it at this point if you are curious. You just need to rename the json file with the same name as the image it corresponds to. 

Then, continuing the workflow, compare the detected WH with my labeled WH:

In [ ]:
#comparison of my labels with the detected WH:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.compare_detections_to_ground_truth(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Create the confusion matrix which summarises the results:

In [ ]:
#create the confusion matrix
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.confusion_matrix_AF(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Extract the recall and the precision of the confusion matrix. Modify the matrix input values manually. 

In [ ]:
import numpy as np
import pandas as pd

# Confusion Matrix
confusion_matrix = np.array([
    [0, 191, 42, 72, 7, 0],
    [38, 32, 1, 7, 3, 0],
    [39, 3, 45, 5, 0, 0],
    [67, 4, 16, 34, 0, 0],
    [7, 2, 3, 4, 13, 0],
    [0, 0, 0, 0, 0, 0]
])

# Class labels
classes = ['Not Classified', 'Dry_WH', 'WH_swamp', 'WH_wet', 'WH_sink', 'U']

def calculate_precision(confusion_matrix):
    precisions = []
    for i in range(len(confusion_matrix)):
        true_positives = confusion_matrix[i][i]
        false_positives = np.sum(confusion_matrix[:, i]) - true_positives
        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
        precisions.append(precision)
    return precisions

def calculate_recall(confusion_matrix):
    recalls = []
    for i in range(len(confusion_matrix)):
        true_positives = confusion_matrix[i][i]
        false_negatives = np.sum(confusion_matrix[i, :]) - true_positives
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
        recalls.append(recall)
    return recalls

# Calculate precision and recall
precision_values = calculate_precision(confusion_matrix)
recall_values = calculate_recall(confusion_matrix)

# Create DataFrames
precision_df = pd.DataFrame({
    'Class': classes,
    'Precision': precision_values
})

recall_df = pd.DataFrame({
    'Class': classes,
    'Recall': recall_values
})

# Print tables
print("Precision Table:")
print(precision_df.to_string(index=False, float_format='{:.3f}'.format))
print("\nRecall Table:")
print(recall_df.to_string(index=False, float_format='{:.3f}'.format))

# Optional: Create a combined table
combined_df = pd.DataFrame({
    'Class': classes,
    'Precision': precision_values,
    'Recall': recall_values
})
print("\nCombined Precision and Recall Table:")
print(combined_df.to_string(index=False, float_format='{:.3f}'.format))

Possible to process a single images by comparing the detections and labels for a single image, used in a function but not usefull by hand. 

In [ ]:
# #create the confusion matrix
# import counting_boats.boat_utils.testing

# counting_boats.boat_utils.testing.process_image_AF(r"D:\Waterholes_project\counting_waterholes\testing_v3\classifications", r"D:/Waterholes_project/counting_waterholes/testing_v3/labels", "config_test_Drive.yaml")

Compare the counts one by one:

In [ ]:
#comparison labelled vs detected WH:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.waterholes_count_compare(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Now we want to plot the waterholes on the images. If you want you can stitch the image separatelly but it is also integrated directly in the next plotting function. 

In [ ]:
# import counting_boats.boat_utils.stitch_PNGs

# counting_boats.boat_utils.stitch_PNGs.stitch_AF(r"D:\Waterholes_project\counting_waterholes\testing_v4\segmented_images\04_06_2024\20240604_mimal_test",
#                                                 r"D:\Waterholes_project\counting_waterholes\testing_v4\stitching")

Now I want to plot the waterholes, modified the function that used to plot boats. Now working on it to plot the waterholes allowing for a comparison of the detected vs my labeled and highlighting if they match. 

In [ ]:
#plot WH after stiching the images:

import counting_boats.boat_utils.stitch_PNGs
import counting_boats.boat_utils.testing
# from counting_boats.boat_utils.stitch_PNGs import stitch_AF

counting_boats.boat_utils.testing.plot_waterholes(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")